# Instance Segmentation

Instance segmentation separates **every object of a scene**: one point mask per object, with a class and a score. Predictions are packed like detections: `masks` $(K, N)$ over the points of the batch, plus `labels`, `scores` and a `batch` index $(K,)$ naming the scene each instance belongs to.

ScanNet, the standard indoor benchmark, scores **18 object classes**: walls and floors are not instances.

## Run a pretrained checkpoint

The example reads one scene of the [`ScanNet20`](../api/datasets/scannet.md) validation split in its scan frame, with the mesh superpoints of ScanNet (`return_superpoint=True` reads the raw `*_vh_clean_2.0.010000.segs.json` files). The dataset requires accepting the ScanNet terms of use: see its [download page](https://kaldir.vc.in.tum.de/scannet/) and extract it under `data/ScanNet/raw/`.

Pass `return_info=True` to get the checkpoint's transform: it voxelizes the scene at 2 cm and keeps `superpoint` and the point-to-voxel `inverse` map at point resolution. `forward` returns the raw query predictions and `decode` turns them into point masks. Score thresholds and minimum mask sizes belong to the evaluation protocol.

In [ ]:
import torch

import torch_pointcloud as tp
from torch_pointcloud.datasets import ScanNet20
from torch_pointcloud.utils.data import collate

# Load the pretrained model
model, info = tp.create_model(
    "spformer.scannet.jiahao-sun",
    task="instance-segmentation",
    pretrained=True,
    return_info=True,
)
model = model.cuda().eval()

# Load one scene of the ScanNet validation split with its superpoints
dataset = ScanNet20(
    root="data",
    split="val",
    use_axis_alignment=False,
    return_superpoint=True,
    transform=info["transform"],
)
sample = dataset[0]

# Collate the sample into a batch
data = collate([sample])
data = {k: v.cuda() if torch.is_tensor(v) else v for k, v in data.items()}
print(f"Voxels: {data['x'].shape[0]}, points: {data['superpoint'].shape[0]}")

# Inference pass
with torch.no_grad():
    out = model(data["x"], data["pos_grid"], data["batch"], data["superpoint"], data["inverse"])

# Decode the queries into point masks
instances = model.decode(out)
masks, labels, scores = instances["masks"], instances["labels"], instances["scores"]
print(f"Masks shape: {tuple(masks.shape)}")

# Report the first confident ones
classes = info["weights"]["classes"]
confident = torch.nonzero(scores > 0.5).flatten()
print(f"Instances scored above 0.5: {len(confident)}")
for i in confident[:8].tolist():
    print(f"{classes[labels[i]]:>15}  {scores[i]:.2f}  {int(masks[i].sum()):>6} points")

# Voxels: 164743, points: 237360
# Masks shape: (100, 237360)
# Instances scored above 0.5: 27
#           chair  0.97    3885 points
#           chair  0.95    2994 points
#  otherfurniture  0.92    1978 points
#           chair  0.90    3092 points
#           chair  0.89    3263 points
#           chair  0.87    3757 points
#           chair  0.89    2372 points
#           chair  0.86    2758 points

SPFormer predicts 400 queries per scene, each a class distribution, a mask quality score and a mask over the superpoints. `decode` keeps the 100 best (query, class) pairs and expands their superpoint masks to the points, so most of the 100 instances have a low score: the benchmark protocol scores them all, ranked by `scores`.

## Inputs and outputs

| Argument     | Shape                    | Description                                                         |
| ------------ | ------------------------ | ------------------------------------------------------------------- |
| `x`          | $(V, 6)$                 | Voxel features: colors in $[-1, 1]$ and centered positions          |
| `pos_grid`   | $(V, 3)$                 | Integer voxel coordinates                                           |
| `batch`      | $(V,)$                   | Index tensor associating each voxel with its scene                  |
| `superpoint` | $(N,)$                   | Superpoint id of every point within its scene                       |
| `inverse`    | $(N,)$                   | Voxel row of every point                                            |
| **returns**  | dict of tensors          | The raw query predictions, which `model.decode(...)` turns into masks |

`decode` returns a packed dict: boolean `masks` $(K, N)$ over the points, `labels` $(K,)$, `scores` $(K,)$ and a `batch` index $(K,)$. These are the inputs of [`instance_matches`](../api/metrics/instance_segmentation.md), which scores one scene against its `instance` and `segment` labels.

## Collating a batch

`superpoint` and `inverse` stay at point resolution while `x` and `pos_grid` are per voxel. Superpoint ids may repeat across scenes, the model keys them by scene. `inverse` has to address the voxel rows of the packed batch: collate it with `cat_keys` to get its `batch_inverse` index, then shift it with `offset_index`.

In [ ]:
from torch_pointcloud.utils.data import DataKeys, PointCloudDataLoader, offset_index

dataloader = PointCloudDataLoader(dataset, batch_size=2, cat_keys=[DataKeys.INVERSE])
data = next(iter(dataloader))
inverse = offset_index(data["inverse"], data["batch_inverse"], data["batch"])

with torch.no_grad():
    out = model(
        data["x"].cuda(),
        data["pos_grid"].cuda(),
        data["batch"].cuda(),
        data["superpoint"].cuda(),
        inverse.cuda(),
    )
instances = model.decode(out)
print(f"Masks shape: {tuple(instances['masks'].shape)}, scenes: {instances['batch'].unique().tolist()}")

# Masks shape: (200, 476621), scenes: [0, 1]

## Evaluate on a dataset

The scorers live in `torch_pointcloud.metrics`. `instance_matches` reduces each scene to a compact record of counts and intersections, and `instance_average_precision` scores the records of the whole split with the ScanNet protocol: AP averaged over the IoU thresholds $0.5, 0.55, \ldots, 0.9$, plus AP50 and AP25. The transform keeps the full-resolution labels as `origin_instance` and `origin_segment`, with the 18 classes as $0$ to $17$ and everything else as $-1$.

In [ ]:
from tqdm.auto import tqdm

from torch_pointcloud.metrics import instance_average_precision, instance_matches

dataloader = PointCloudDataLoader(dataset, batch_size=1, num_workers=4)

matches = []
with torch.no_grad():
    for data in tqdm(dataloader, desc="Evaluating"):
        data = {k: v.cuda() if torch.is_tensor(v) else v for k, v in data.items()}
        out = model(data["x"], data["pos_grid"], data["batch"], data["superpoint"], data["inverse"])
        instances = model.decode(out)
        masks, labels, scores = instances["masks"], instances["labels"], instances["scores"]

        keep = (scores > 0) & (masks.sum(dim=1) > 100)
        matches.append(
            instance_matches(
                masks[keep],
                labels[keep],
                scores[keep],
                data["origin_instance"],
                data["origin_segment"],
            )
        )

mean_ap = instance_average_precision(matches, num_classes=18)
ap50 = instance_average_precision(matches, iou_threshold=0.5, num_classes=18)
ap25 = instance_average_precision(matches, iou_threshold=0.25, num_classes=18)
print(f"mAP {mean_ap:.4f} | mAP@0.5 {ap50:.4f} | mAP@0.25 {ap25:.4f}")

# mAP 0.5647 | mAP@0.5 0.7393 | mAP@0.25 0.8255

## Train from scratch

While :pytorch-pointcloud-mini: `torch-pointcloud` provides the models and the utilities around them, you still own the training loop. `SPFormerLoss` matches the queries to the ground-truth instances with a Hungarian assignment and supervises every decoder layer; it reads the instance and class labels at the resolution of `superpoint`. The loop below runs one demo epoch with the evaluation transform; `examples/spformer/instance_segmentation_scannet_train.py` follows the authors' recipe, augmentations included.

In [ ]:
from torch_pointcloud.losses import SPFormerLoss

device = "cuda"

train_dataset = ScanNet20(
    root="data",
    split="train",
    use_axis_alignment=False,
    return_superpoint=True,
    transform=info["transform"],
)
train_dataloader = PointCloudDataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=8,
    cat_keys=[DataKeys.INVERSE],
)

# Create the desired model, criterion and optimizer
model = tp.create_model("spformer.scannet.jiahao-sun", task="instance-segmentation").to(device)
# The transform keeps the labels at point resolution, the resolution of `superpoint`, as `origin_*`.
criterion = SPFormerLoss(
    num_classes=18,
    instance_key=DataKeys.ORIGIN_INSTANCE,
    label_key=DataKeys.ORIGIN_SEGMENT,
).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.05)

# Training loop
num_epochs = 1  # a demo pass, raise it to actually train

model.train()
for epoch in range(num_epochs):
    pbar = tqdm(train_dataloader, desc=f"Epoch {epoch}")
    for data in pbar:
        data = {k: v.to(device) if torch.is_tensor(v) else v for k, v in data.items()}
        inverse = offset_index(data["inverse"], data["batch_inverse"], data["batch"])
        out = model(data["x"], data["pos_grid"], data["batch"], data["superpoint"], inverse)
        losses = criterion(out, data)

        optimizer.zero_grad()
        losses["loss"].backward()
        optimizer.step()
        pbar.set_postfix(loss=f"{losses['loss'].item():.3f}")